# Climate SLM: evaluation runner (Colab, T4 GPU)

Run the cells from top to bottom. Before you start: menu **Runtime > Change runtime type > T4 GPU**.

There is no logic in this notebook. Every cell only calls code from the bundle.

**Step 1.** Check that a GPU is there.

In [ ]:
# 1. GPU check: print the GPU name, stop with a clear message if there is no GPU
import torch
if not torch.cuda.is_available():
    raise SystemExit("No GPU found. Menu: Runtime > Change runtime type > T4 GPU. Then run this cell again.")
print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__)

**Step 2.** Upload the bundle. Click the **folder icon** on the left, then the **upload** button, and choose `colab_bundle.zip` (made on the laptop by `python scripts/make_colab_bundle.py`). Wait until the upload is finished, then run the cell.

In [ ]:
# 2. Unzip the uploaded bundle into /content/climate_slm_project and go there
import os, zipfile
if not os.path.exists("/content/colab_bundle.zip"):
    raise SystemExit("colab_bundle.zip not found in /content. Upload it with the folder icon on the left.")
zipfile.ZipFile("/content/colab_bundle.zip").extractall("/content/climate_slm_project")
%cd /content/climate_slm_project
!ls

**Step 3.** Install the extra packages. torch is not in the list, so Colab's own torch stays.

In [ ]:
# 3. Install packages from requirements-gpu.txt (torch is not reinstalled)
!pip install -q -r requirements-gpu.txt
!python -c "import torch, transformers; print('torch', torch.__version__, '| transformers', transformers.__version__)" 

**Step 4.** Smoke test: 3 test items only. Read the three outputs. If this fails, stop and send the error text.

In [ ]:
# 4. Smoke test: base model on the first 3 test items, prints the 3 outputs
!python -m climate_slm.evaluation.evaluate --model base --split test --limit 3 --out outputs/smoke_base_test.json

**Step 5.** Full runs. This is the real baseline.

In [ ]:
# 5. Full runs: base model on val, then on test
!python -m climate_slm.evaluation.evaluate --model base --split val
!python -m climate_slm.evaluation.evaluate --model base --split test

**Step 6 (Phase 2c).** Stronger prompts on the same base model. 4 runs, about 5 minutes (estimate). The names of the files you get:
`eval_base_p_sentence_val.json`, `eval_base_p_sentence_test.json`, `eval_base_p_fewshot_val.json`, `eval_base_p_fewshot_test.json`.

In [ ]:
# 6. Prompt variants: base model with the strict_sentence prompt and the fewshot prompt, on val and test
!python -m climate_slm.evaluation.evaluate --model base --split val --prompt-variant strict_sentence
!python -m climate_slm.evaluation.evaluate --model base --split test --prompt-variant strict_sentence
!python -m climate_slm.evaluation.evaluate --model base --split val --prompt-variant fewshot
!python -m climate_slm.evaluation.evaluate --model base --split test --prompt-variant fewshot

**Step 7 (Phase 2c).** Decoding check: the base model with Qwen's own sampling settings, 3 seeds, val and test. 6 runs, about 6 minutes (estimate).
Files: `eval_base_qwen_s1_val.json` ... `eval_base_qwen_s3_test.json` (6 files).

In [ ]:
# 7. Decoding check: base model with Qwen default sampling, seeds 1, 2, 3, on val and test
!python -m climate_slm.evaluation.evaluate --model base --split val --decoding qwen_default --seed 1
!python -m climate_slm.evaluation.evaluate --model base --split test --decoding qwen_default --seed 1
!python -m climate_slm.evaluation.evaluate --model base --split val --decoding qwen_default --seed 2
!python -m climate_slm.evaluation.evaluate --model base --split test --decoding qwen_default --seed 2
!python -m climate_slm.evaluation.evaluate --model base --split val --decoding qwen_default --seed 3
!python -m climate_slm.evaluation.evaluate --model base --split test --decoding qwen_default --seed 3

**Step 8.** Download the new results (only the Step 6 and Step 7 files, so the first base measurement is never overwritten). Unzip the file into the `outputs/` folder on the laptop.

In [ ]:
# 8. Zip the new result files (prompt variants and seeds, with their review CSVs) and download the zip
!python -m climate_slm.evaluation.collect --out /content/eval_results_2c.zip
from google.colab import files
files.download("/content/eval_results_2c.zip")

# Phase 3: SFT with LoRA and QLoRA

Run the cells in this order. Cells 1 to 3 of the evaluation part (GPU check, upload and unzip the bundle, install) must have been run in this session first.
Training cells skip runs that are already finished, so after a disconnect you can run the same cell again.
Times are ESTIMATES (from the size of the job, not measured): grid training 25 to 35 min, val evaluations 15 min, seed repeats 15 min, 1.5B arm 30 to 40 min, install 2 min. Plan 1.5 to 2 hours in total, over one or two sessions.
Only cell P3-9 touches the test split.

**P3-1.** Remove torchao (the old Colab copy breaks peft, and we do not use it), install the extra packages and print the versions that were installed.

In [ ]:
# P3-1. Install the Phase 3 packages (torch is not reinstalled), print the versions and the GPU
!pip uninstall -y -q torchao
!pip install -q -r requirements-gpu.txt
import importlib.metadata as md, torch
print({p: md.version(p) for p in ["torch", "transformers", "accelerate", "peft", "bitsandbytes", "mlflow"]})
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), "GB")

**P3-2.** Train the 4 grid runs on the 0.5B model (LoRA first, then QLoRA). Files: `outputs/adapters/<run>/e1..e3`, `outputs/runs/<run>/train_log.json`, `outputs/mlruns`. If a QLoRA run fails, the others still run; the error is in `outputs/runs/<run>/train_error.txt`.

In [ ]:
# P3-2. Train the 4 grid runs on 0.5B (finished runs are skipped)
!python -m climate_slm.training.train --group grid

**P3-3.** Evaluate all 12 epoch adapters on the VALIDATION set (the cell cannot evaluate test). Files: `outputs/eval_<run>_e<k>_val.json`.

In [ ]:
# P3-3. Evaluate every epoch adapter of the grid on VAL and log the numbers to MLflow
!python -m climate_slm.training.eval_adapters --group grid --split val

**P3-3b.** Evaluate the 6 QLoRA adapters of the grid on the 4-bit base they were trained on (VAL only, cannot evaluate test). These are EXTRA rows: files `eval_<run>_e<k>__on4bit_val.json`; the fp16-base results of P3-3 stay as they are. Run it after P3-3 and before P3-4. About 10 minutes (estimate).

In [ ]:
# P3-3b. Evaluate the QLoRA adapters on the 4-bit base they were trained on (VAL only, extra rows)
!python -m climate_slm.training.eval_adapters --group grid --split val --on-4bit-base

**P3-4.** Apply the selection rule (fixed before the runs, on val only). Writes `outputs/phase3_selection.md` and `.json`. Read the table before going on.

In [ ]:
# P3-4. Select the winner on VAL only (writes outputs/phase3_selection.md)
!python -m climate_slm.training.select_best

**P3-5.** Repeat the winning configuration with seeds 43 and 44 and evaluate those adapters on val.

In [ ]:
# P3-5. Repeat the winner with seeds 43 and 44, evaluate the new adapters on VAL
!python -m climate_slm.training.train --group repeats
!python -m climate_slm.training.eval_adapters --group repeats --split val

**P3-6.** The 1.5B arm: one run with the winning method and learning rate (Qwen/Qwen2.5-1.5B-Instruct), then its epochs on val. Gradient checkpointing is on for this model.

In [ ]:
# P3-6. 1.5B arm: untuned 1.5B on VAL, one tuned 1.5B run, its epoch adapters on VAL
!python -m climate_slm.evaluation.evaluate --model base --base-model-id Qwen/Qwen2.5-1.5B-Instruct --name base15 --split val
!python -m climate_slm.training.train --group q15
!python -m climate_slm.training.eval_adapters --group q15 --split val

**P3-7.** The UNTUNED 1.5B model on the test split (no selection is involved; this is a baseline, like the 0.5B base in Phase 2).

In [ ]:
# P3-7. Untuned 1.5B baseline on the test split (nothing is tuned or selected here)
!python -m climate_slm.evaluation.evaluate --model base --base-model-id Qwen/Qwen2.5-1.5B-Instruct --name base15 --split test

**P3-8.** Select again (now with the seed repeats and the 1.5B arm) and FREEZE the selection. After this cell the selection file is not rewritten.

In [ ]:
# P3-8. Final val selection, then freeze it (stores the sha256 of outputs/phase3_selection.json)
!python -m climate_slm.training.select_best --freeze

**P3-9. FINAL TEST.** Run only after P3-8 and only once. Each winner is evaluated on the test split exactly once; the cell refuses to run if the selection is not frozen or a winner was tested before.

In [ ]:
# P3-9. FINAL TEST: the frozen winners on the test split, once each, then the val+test tables
!python -m climate_slm.training.final_test --size 0.5b
!python -m climate_slm.training.final_test --size 1.5b
!python -m climate_slm.training.select_best --final-tables

**P3-10.** Download. `phase3_results.zip` is small (tables, eval files, run logs, MLflow). `phase3_adapters.zip` has the adapters (large, about 0.9 GB). Unzip both into `outputs/` on the laptop.

In [ ]:
# P3-10. Zip the results (small) and the adapters (large), then download both
!python -m climate_slm.evaluation.collect --set phase3-results --out /content/phase3_results.zip
!python -m climate_slm.evaluation.collect --set phase3-adapters --out /content/phase3_adapters.zip
from google.colab import files
files.download("/content/phase3_results.zip")
files.download("/content/phase3_adapters.zip")

# Phase 4: DPO on top of the SFT model (0.5B)

New session: run set-up cells 1 (GPU check), 2 (upload the NEW `colab_bundle.zip` and unzip) and 3 (install) first. P4-0 (pairs input) is done; run it again only if `eval_base_train.json` is lost. Then P4-1 to P4-10 in this order. Cells P4-3 to P4-8 use the VAL split only; the one test run is P4-9.

**P4-0.** Run the UNTUNED 0.5B on the TRAIN split (greedy, default prompt, same evaluator). This gives the wrong answers that become the "rejected" side of the DPO pairs. The evaluator refuses `--split train` for any tuned model. Download `eval_base_train.json` and put it into `outputs/` on the laptop; the pairs are built there (no GPU needed).

In [ ]:
# P4-0. Untuned 0.5B on the train split (input of the DPO pair builder), then download the file
!python -m climate_slm.evaluation.evaluate --model base --split train
from google.colab import files
files.download("/content/climate_slm_project/outputs/eval_base_train.json")

**P4-1.** Remove torchao, install the packages and print the versions and the GPU.

In [ ]:
# P4-1. Install the packages (torch is not reinstalled), print the versions and the GPU
!pip uninstall -y -q torchao
!pip install -q -r requirements-gpu.txt
import importlib.metadata as md, torch
print({p: md.version(p) for p in ["torch", "transformers", "accelerate", "peft", "mlflow"]})
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), "GB")

**P4-2.** Upload `phase4_start.zip` (made on the laptop: the three SFT start adapters and their val files, about 93 MB) with the folder icon, then run the cell. It unzips into `outputs/` and checks the pairs file, the SFT adapters (sha256) and the config. Stop here if it prints a problem.

In [ ]:
# P4-2. Unzip the SFT start adapters into outputs/ and check pairs, adapters and config (no GPU needed)
import os, zipfile
if not os.path.exists("/content/phase4_start.zip"):
    raise SystemExit("phase4_start.zip not found in /content. Upload it with the folder icon on the left.")
zipfile.ZipFile("/content/phase4_start.zip").extractall("/content/climate_slm_project/outputs")
!python -m climate_slm.training.dpo_train --check

**P4-3.** Train the 4 DPO runs (beta 0.1 and 0.3 x learning rate 5e-5 and 2e-4, seed 42, 3 epochs, one adapter per epoch). Each run is its own process; a finished run is skipped, so you can run the cell again after a disconnect.

In [ ]:
# P4-3. DPO grid: 4 runs, each starts from the SFT adapter (seed 42), adapters saved after every epoch
!python -m climate_slm.training.dpo_train --group grid

**P4-4.** Evaluate the 12 epoch adapters on the VAL split (same evaluator as every other model) and log to MLflow.

In [ ]:
# P4-4. Evaluate the 12 DPO adapters on VAL (greedy, same evaluator as all other models)
!python -m climate_slm.training.dpo_eval --group grid --split val

**P4-5.** Apply the fixed selection rule on VAL and print the result. If it says "no DPO candidate improves on the SFT model on val", stop after P4-8: the test cell will refuse and SFT stays.

In [ ]:
# P4-5. Selection on VAL: candidates, SFT control, winner or "no DPO candidate improves"
!python -m climate_slm.training.select_dpo
!cat outputs/phase4_selection.md

**P4-6.** Repeat ONLY the winning configuration with seeds 43 and 44 (each starts from the SFT adapter of its own seed). Does nothing if there is no winner.

In [ ]:
# P4-6. Seed repeats of the winning DPO configuration (seeds 43 and 44)
!python -m climate_slm.training.dpo_train --group repeats

**P4-7.** Evaluate the repeat adapters on the VAL split.

In [ ]:
# P4-7. Evaluate the repeat adapters on VAL
!python -m climate_slm.training.dpo_eval --group repeats --split val

**P4-8.** Select again (now with the paired seeds) and FREEZE the selection. After this cell the selection file is not rewritten.

In [ ]:
# P4-8. Final val selection with the paired-seed table, then freeze it (stores the sha256 of phase4_selection.json)
!python -m climate_slm.training.select_dpo --freeze
!cat outputs/phase4_selection.md

**P4-9. FINAL TEST.** Run only after P4-8 and only once. The frozen DPO winner is evaluated on the test split exactly once; the cell refuses to run if the selection is not frozen, has no winner, or this candidate was tested before. The SFT control is not run again (its test result already exists on the laptop).

In [ ]:
# P4-9. FINAL TEST: the frozen DPO winner on the test split, once
!python -m climate_slm.training.dpo_final_test

**P4-10.** Download. `phase4_results.zip` is small (selection files, eval files, run logs with the reference log-probs, MLflow). `phase4_adapters.zip` has the DPO adapters (about 0.4 GB). Unzip both into `outputs/` on the laptop, then run `python -m climate_slm.training.select_dpo --final-tables` there.

In [ ]:
# P4-10. Zip the results (small) and the DPO adapters (large), then download both
!python -m climate_slm.evaluation.collect --set phase4-results --out /content/phase4_results.zip
!python -m climate_slm.evaluation.collect --set phase4-adapters --out /content/phase4_adapters.zip
from google.colab import files
files.download("/content/phase4_results.zip")
files.download("/content/phase4_adapters.zip")

# Phase 4b: DPO v2 (stage A: rejected texts for the pairs)

Only after the reviewer says go. New session: set-up cells 1 to 3, then P4-1 and P4-2 (they unzip the SFT start adapter and check its hash). The pairs are built on the laptop afterwards.

**P4b-0.** Generate the candidate rejected texts for the 21 refusal prompts of the TRAIN split: the untuned 0.5B (4 sampled outputs) and the SFT 0.5B (greedy + 4 sampled outputs), fixed seeds 42 to 45. This is not an evaluation. Download `dpo2_candidates.json` into `outputs/` on the laptop and run `python -m climate_slm.training.dpo2_pairs --show`.

In [ ]:
# P4b-0. Rejected-text candidates for the DPO v2 pairs (train refusal prompts only), then download the file
!python -m climate_slm.training.dpo2_generate
from google.colab import files
files.download("/content/climate_slm_project/outputs/dpo2_candidates.json")

## Phase 4b, stage B: DPO v2 training and evaluation

New session: set-up cells 1 (GPU check), 2 (upload the NEW `colab_bundle.zip`), 3 (install). Then P4b-1 to P4b-10 in this order. P4b-0 (candidate texts) is done. Cells P4b-3 to P4b-8 use the VAL split only; the one test run is P4b-9. The v1 cells P4-1 to P4-10 are not needed.

**P4b-1.** Remove torchao, install the packages and print the versions and the GPU.

In [ ]:
# P4b-1. Install the packages (torch is not reinstalled), print the versions and the GPU
!pip uninstall -y -q torchao
!pip install -q -r requirements-gpu.txt
import importlib.metadata as md, torch
print({p: md.version(p) for p in ["torch", "transformers", "accelerate", "peft", "mlflow"]})
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), "GB")

**P4b-2.** Upload `phase4_start.zip` (the same file as for v1: the three SFT start adapters and their val files, about 93 MB), then run the cell. It unzips into `outputs/` and checks the config, both pair files (sha256, number of steps) and the SFT adapters (sha256). Stop here if it prints a problem.

In [ ]:
# P4b-2. Unzip the SFT start adapters into outputs/ and check config, pair files and adapters (no GPU work)
import os, zipfile
if not os.path.exists("/content/phase4_start.zip"):
    raise SystemExit("phase4_start.zip not found in /content. Upload it with the folder icon on the left.")
zipfile.ZipFile("/content/phase4_start.zip").extractall("/content/climate_slm_project/outputs")
!python -m climate_slm.training.dpo2_train --check

**P4b-3.** Train the 8 DPO v2 runs (pair set v2-main / v2-mix x alpha 0.5 / 1.0 x learning rate 1e-5 / 2e-5; beta 0.1, seed 42, effective batch 8, 3 epochs, one adapter per epoch). Each run is its own process; a finished run is skipped, so you can run the cell again after a disconnect.

In [ ]:
# P4b-3. DPO v2 grid: 8 runs, each starts from the SFT adapter (seed 42), adapters saved after every epoch
!python -m climate_slm.training.dpo2_train --group grid

**P4b-4.** Evaluate the 24 epoch adapters on the VAL split (same evaluator as every other model) and log to MLflow.

In [ ]:
# P4b-4. Evaluate the 24 DPO v2 adapters on VAL (greedy, same evaluator as all other models)
!python -m climate_slm.training.dpo2_eval --group grid --split val

**P4b-5.** Apply the fixed v2 selection rule on VAL and print the result. If it says "no DPO v2 candidate improves on the SFT model on val", run P4b-8 to record it and stop: P4b-9 will refuse and SFT stays.

In [ ]:
# P4b-5. Selection on VAL: 24 candidates, SFT control, winner or "no DPO v2 candidate improves"
!python -m climate_slm.training.select_dpo2
!cat outputs/phase4b_selection.md

**P4b-6.** Repeat ONLY the winning configuration with seeds 43 and 44 (each starts from the SFT adapter of its own seed). Does nothing if there is no winner.

In [ ]:
# P4b-6. Seed repeats of the winning DPO v2 configuration (seeds 43 and 44)
!python -m climate_slm.training.dpo2_train --group repeats

**P4b-7.** Evaluate the repeat adapters on the VAL split.

In [ ]:
# P4b-7. Evaluate the repeat adapters on VAL
!python -m climate_slm.training.dpo2_eval --group repeats --split val

**P4b-8.** Select again (now with the paired seeds) and FREEZE the selection. After this cell the selection file is not rewritten.

In [ ]:
# P4b-8. Final val selection with the paired-seed table, then freeze it (stores the sha256 of phase4b_selection.json)
!python -m climate_slm.training.select_dpo2 --freeze
!cat outputs/phase4b_selection.md

**P4b-9. FINAL TEST.** Run only after P4b-8 and only once. The frozen DPO v2 winner is evaluated on the test split exactly once; the cell refuses to run if the selection is not frozen, has no winner, or this candidate was tested before. The SFT control is not run again (its test result already exists on the laptop).

In [ ]:
# P4b-9. FINAL TEST: the frozen DPO v2 winner on the test split, once
!python -m climate_slm.training.dpo2_final_test

**P4b-10.** Download. `phase4b_results.zip` is small (selection files, eval files, run logs with the reference log-probs, MLflow). `phase4b_adapters.zip` has the DPO v2 adapters (about 0.8 GB for 24 + 6 adapters). Unzip both into `outputs/` on the laptop, then run `python -m climate_slm.training.select_dpo2 --final-tables` there.

In [ ]:
# P4b-10. Zip the results (small) and the DPO v2 adapters (large), then download both
!python -m climate_slm.evaluation.collect --set phase4b-results --out /content/phase4b_results.zip
!python -m climate_slm.evaluation.collect --set phase4b-adapters --out /content/phase4b_adapters.zip
from google.colab import files
files.download("/content/phase4b_results.zip")
files.download("/content/phase4b_adapters.zip")

# Phase 5: serving the tuned model (vLLM behind FastAPI)

**Stage 1 only: cell P5-0.** vLLM brings its own torch and can break the training environment, so this section needs a FRESH runtime: menu **Runtime > Restart session** (or a new session), then run ONLY set-up cell 2 (upload and unzip the NEW `colab_bundle.zip`; it also does `%cd`). Do NOT run set-up cell 3 or any Phase 3 / 4 cell in this runtime. Upload `phase4_start.zip` (the same file as for Phase 4: it holds the 0.5B SFT adapter and its saved val outputs). The later Phase 5 cells (service, parity check, benchmark) are written after the result of P5-0 is known.

**P5-0.** Install vLLM in this fresh runtime, print the versions pip chose, load the base model in vLLM with `--dtype half`, serve the 0.5B SFT adapter with vLLM's LoRA support and answer 3 val prompts greedily. Every step prints PASS or FAIL; the result is also written to `outputs/phase5_smoke.json`. If a step fails, the hf backend is used instead and every sentence about vLLM is removed from the results. Send the printed output (and the json) back to the reviewer before anything else is written.

In [ ]:
%cd /content/climate_slm_project
# P5-0. vLLM install check in a FRESH runtime (restart first; only set-up cell 2 before this). Training packages are not touched.
import os, zipfile
if not os.path.exists("/content/phase4_start.zip"):
    raise SystemExit("phase4_start.zip not found in /content. Upload it with the folder icon on the left.")
zipfile.ZipFile("/content/phase4_start.zip").extractall("/content/climate_slm_project/outputs")
!pip install -q vllm==0.32.0
!pip list 2>/dev/null | grep -i -E "^(vllm|torch|transformers|xformers|flashinfer-python|triton) "
!python scripts/vllm_smoke.py

## Phase 5, stage 2: parity check and benchmark (cells P5-1 to P5-6; written, NOT run yet)

Same fresh vLLM runtime as P5-0 (P5-0 passed: vLLM 0.32.0, LoRA works, 3 of 3 outputs identical). Do NOT run set-up cell 3 or any Phase 3 / 4 cell here. Every cell starts with `%cd /content/climate_slm_project`. Upload the NEW `colab_bundle.zip` (set-up cell 2) and `phase5_start.zip` (three adapters, about 127 MB). Order: P5-1, P5-2 (parity), and ONLY if P5-2 does not print STOP: P5-3, P5-4, P5-5, P5-6. All cells use the val split only.

**P5-1.** Unzip the three adapters and their saved val files into `outputs/`, then check files, GPU, library versions and whether the hf backend can run in this runtime (transformers and peft importable). Nothing is run on the GPU.

The cell also installs `rouge_score` (the evaluator's scoring code needs it for ROUGE-L; without it the parity check crashed) and removes `torchao` (Colab's copy 0.10.0 is too old for peft 0.21.2, so the hf backend could not start). Removing torchao is only for this vLLM runtime; the training cells are not touched.

In [ ]:
%cd /content/climate_slm_project
# P5-1. Unzip the three adapters (upload phase5_start.zip first) and check files, GPU and versions
import os, zipfile
if not os.path.exists("/content/phase5_start.zip"):
    raise SystemExit("phase5_start.zip not found in /content. Upload it with the folder icon on the left.")
zipfile.ZipFile("/content/phase5_start.zip").extractall("/content/climate_slm_project/outputs")
!pip install -q httpx
!pip install -q rouge_score
!pip uninstall -y torchao
!python scripts/phase5_check.py

**P5-2. PARITY CHECK.** For each of the three models (0.5B SFT, 0.5B DPO v2, 1.5B SFT) a vLLM service is started, all 53 val prompts are sent one at a time, the answers are scored with the evaluator's own scoring code and compared with the saved evaluator val file of the same adapter: number of identical outputs, answerable token F1, strict correct refusals, false refusals. Rule fixed in advance: if strict correct refusals or false refusals move by more than 1 item, or F1 by more than 0.03, the cell prints **STOP**. Then do NOT run the benchmark: send the output (and `outputs/phase5/parity_*.json`) to the reviewer.

In [ ]:
%cd /content/climate_slm_project
# P5-2. PARITY CHECK: the live vLLM service against the saved evaluator val files, 3 models x 53 prompts
!python scripts/parity_check.py

**P5-3.** Benchmark, vLLM: for each model a service is started (its start-up time is measured separately), 3 warm-up requests, then concurrency 1, 4 and 16, 3 repeats each, the 53 val prompts per repeat. It refuses to run for a model whose parity check did not pass. A finished setting is skipped if you run the cell again (a disconnect loses at most one setting).

In [ ]:
%cd /content/climate_slm_project
# P5-3. BENCHMARK vLLM: latency p50 / p95, tokens per second, requests per second, GPU memory; concurrency 1, 4, 16; 3 repeats
!python scripts/bench_serving.py --backends vllm

**P5-4.** The same benchmark for the hf backend (the evaluator's own transformers code behind the same service). If transformers or peft cannot be imported in this runtime, the settings are written as skipped with the reason and reported as not measured.

In [ ]:
%cd /content/climate_slm_project
# P5-4. BENCHMARK hf: the same settings, the same prompts, the same GPU
!python scripts/bench_serving.py --backends hf

**P5-5.** Make the tables from the saved files (parity, start-up times, benchmark with min / median / max over the repeats, where two settings are too close to separate) and print them.

In [ ]:
%cd /content/climate_slm_project
# P5-5. Tables from the saved result files (no number is typed by hand)
!python scripts/phase5_report.py
!cat outputs/phase5_results.md

**P5-6.** Download `phase5_results.zip` (tables, csv, raw results, server logs, evaluator-format files of the live service). Unzip it into `outputs/` on the laptop. MLflow (experiment `phase5_serving`) is logged on the laptop from these files: `python scripts/phase5_report.py --log-mlflow` (mlflow is not installed in the vLLM runtime on purpose).

In [ ]:
%cd /content/climate_slm_project
# P5-6. Zip the Phase 5 results (small) and download them
!python -m climate_slm.evaluation.collect --set phase5-results --out /content/phase5_results.zip
from google.colab import files
files.download("/content/phase5_results.zip")